# FCC ↔ BCC Solid–Solid Transition in Iron

Iron undergoes a structural phase transition from BCC (α-Fe) to FCC (γ-Fe) at approximately 1185 K.
Despite BCC iron having a lower potential energy at 0 K, the softer phonon spectrum of FCC produces
higher vibrational entropy at elevated temperatures, eventually stabilising the FCC structure.

This notebook demonstrates two approaches to model this transition:

1. **Debye model** — analytic $g(E) \propto E^2$ DOS using Debye temperatures from experiment.
2. **ASE `Phonons` + EMT** — a finite-displacement force-constant calculation, illustrating the full
   ASE phonon workflow.

> **Note on the EMT potential.**  EMT is parameterised for FCC metals (Ni, Cu, …) rather than for
> Fe. The BCC structure is mechanically unstable under pair-like potentials (imaginary modes), whose
> contribution to the DOS is discarded.  The EMT phases therefore serve as a *workflow demonstration*;
> for quantitative Fe results the Debye (or a DFT + phonopy) approach is preferred.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import trapezoid
from scipy.constants import k, eV

from ase.build import bulk
from ase.calculators.emt import EMT
from ase.phonons import Phonons
from ase.thermochemistry import CrystalThermo

from landau.ase_phases import AsePhase
from landau.calculate import calc_phase_diagram
from landau.plot import plot_1d_T_phase_diagram

kB = k / eV  # eV / K

## 1  Debye Model

The Debye model approximates the phonon density of states as

$$g(E) = \frac{3}{E_D^3}\,E^2 \quad \text{for } 0 \le E \le E_D, \qquad E_D = k_B\,\theta_D$$

normalised so that $\int_0^{E_D} g(E)\,dE = 3$ (three modes per atom).

| Phase | Structure | $\theta_D$ (K) | $E_D$ (meV) |
|-------|-----------|--------------|-------------|
| α-Fe  | BCC       | 470           | 40.5        |
| γ-Fe  | FCC       | 390           | 33.6        |

FCC has a lower Debye temperature (softer phonons), giving higher vibrational entropy at elevated
temperatures and hence a lower free energy above the transition.

In [ ]:
theta_bcc = 470.0  # K, alpha-Fe (BCC)
theta_fcc = 390.0  # K, gamma-Fe (FCC)

E_D_bcc = kB * theta_bcc
E_D_fcc = kB * theta_fcc

energies_debye = np.linspace(0.0, max(E_D_bcc, E_D_fcc) * 1.05, 500)


def debye_dos(energies, E_D):
    dos = np.where(energies <= E_D, energies**2, 0.0)
    return dos * 3.0 / trapezoid(dos, energies)


dos_bcc_debye = debye_dos(energies_debye, E_D_bcc)
dos_fcc_debye = debye_dos(energies_debye, E_D_fcc)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(energies_debye * 1e3, dos_bcc_debye, label=rf'α-Fe BCC  ($\theta_D$ = {theta_bcc:.0f} K)')
ax.plot(energies_debye * 1e3, dos_fcc_debye, label=rf'γ-Fe FCC  ($\theta_D$ = {theta_fcc:.0f} K)')
ax.set_xlabel('Phonon energy (meV)')
ax.set_ylabel('DOS (1/eV)')
ax.legend()
ax.set_title('Debye phonon DOS')
plt.tight_layout()

### Phase objects — Debye model

Each phase is wrapped in an `AsePhase` that calls `CrystalThermo` to evaluate the Helmholtz free energy

$$F(T) = E_0 + E_\text{ZPE} + k_BT\int_0^\infty g(E)\,\ln\!\left[2\sinh\!\left(\frac{E}{2k_BT}\right)\right]dE$$

The potential energies $E_0$ are representative DFT values for Fe:
BCC is taken as the reference ($E_0^\text{BCC} = 0$) and FCC lies ~57 meV/atom higher at 0 K.

In [ ]:
E0_bcc = 0.000  # eV/atom, DFT reference
E0_fcc = 0.057  # eV/atom, FCC less stable at T = 0

bcc_debye = AsePhase(
    name='α-Fe (Debye)',
    fixed_concentration=0.0,
    thermochem=CrystalThermo(
        phonon_DOS=dos_bcc_debye,
        phonon_energies=energies_debye,
        formula_units=1,
        potentialenergy=E0_bcc,
    ),
)

fcc_debye = AsePhase(
    name='γ-Fe (Debye)',
    fixed_concentration=0.0,
    thermochem=CrystalThermo(
        phonon_DOS=dos_fcc_debye,
        phonon_energies=energies_debye,
        formula_units=1,
        potentialenergy=E0_fcc,
    ),
)

## 2  Real Phonon Calculations with ASE

The `ase.phonons.Phonons` class computes the force-constant matrix by finite displacements and
diagonalises it on a dense **k**-point mesh to obtain the phonon DOS.  The workflow is:

```python
# 1. Build the atoms object and compute E0 before running phonons
atoms.calc = calc
E0 = atoms.get_potential_energy() / len(atoms)

# 2. Finite-displacement force-constant calculation
ph = Phonons(atoms, calc, supercell=..., delta=..., name=...)  # set up
ph.run()                           # displace each atom and compute forces
ph.read(acoustic=True)             # assemble force constants, enforce sum rule

# 3. Sample the phonon DOS on a k-point mesh
raw_dos = ph.get_dos(kpts=..., verbose=False)          # RawDOSData
grid_dos = raw_dos.sample_grid(npts=..., xmin=0.0, width=...)  # GridDOSData
energies = grid_dos.get_energies()
weights  = grid_dos.get_weights()
```

Setting `xmin=0.0` discards imaginary modes (negative frequencies).  The resulting DOS is normalised
to three modes per atom.

We use EMT with Ni at the experimental Fe lattice constants as a readily available calculator.
A BCC structure is mechanically metastable under EMT, so imaginary modes appear and are discarded.

In [ ]:
# BCC at the experimental Fe BCC lattice constant (a = 2.87 Å)
atoms_bcc = bulk('Ni', crystalstructure='bcc', a=2.87)
atoms_bcc.calc = EMT()
E0_bcc_emt = atoms_bcc.get_potential_energy() / len(atoms_bcc)  # must precede ph.run()

ph_bcc = Phonons(atoms_bcc, EMT(), supercell=(3, 3, 3), delta=0.05, name='fe-bcc-phonon')
ph_bcc.run()
ph_bcc.read(acoustic=True)

raw_dos_bcc = ph_bcc.get_dos(kpts=(20, 20, 20), verbose=False)
grid_dos_bcc = raw_dos_bcc.sample_grid(npts=500, xmin=0.0, width=0.003)
omega_bcc_emt = grid_dos_bcc.get_energies()
dos_bcc_emt = grid_dos_bcc.get_weights()
dos_bcc_emt = dos_bcc_emt * 3.0 / trapezoid(dos_bcc_emt, omega_bcc_emt)

print(f'BCC (EMT Ni at a=2.87 Å)  ω_max = {omega_bcc_emt.max()*1e3:.1f} meV')

In [ ]:
# FCC at the experimental Fe FCC lattice constant (a = 3.64 Å)
atoms_fcc = bulk('Ni', crystalstructure='fcc', a=3.64)
atoms_fcc.calc = EMT()
E0_fcc_emt = atoms_fcc.get_potential_energy() / len(atoms_fcc)

ph_fcc = Phonons(atoms_fcc, EMT(), supercell=(3, 3, 3), delta=0.05, name='fe-fcc-phonon')
ph_fcc.run()
ph_fcc.read(acoustic=True)

raw_dos_fcc = ph_fcc.get_dos(kpts=(20, 20, 20), verbose=False)
grid_dos_fcc = raw_dos_fcc.sample_grid(npts=500, xmin=0.0, width=0.003)
omega_fcc_emt = grid_dos_fcc.get_energies()
dos_fcc_emt = grid_dos_fcc.get_weights()
dos_fcc_emt = dos_fcc_emt * 3.0 / trapezoid(dos_fcc_emt, omega_fcc_emt)

print(f'FCC (EMT Ni at a=3.64 Å)  ω_max = {omega_fcc_emt.max()*1e3:.1f} meV')

### Phase objects — EMT phonons

We wrap the EMT phonon DOS in `AsePhase`, using the same DFT-quality reference energies as the Debye
model so that all four free-energy curves are directly comparable.
The only difference between the Debye and EMT phases is the shape of the phonon DOS.

In [ ]:
bcc_emt = AsePhase(
    name='α-Fe (EMT phonons)',
    fixed_concentration=0.0,
    thermochem=CrystalThermo(
        phonon_DOS=dos_bcc_emt,
        phonon_energies=omega_bcc_emt,
        formula_units=1,
        potentialenergy=E0_bcc,  # same DFT reference as Debye BCC
    ),
)

fcc_emt = AsePhase(
    name='γ-Fe (EMT phonons)',
    fixed_concentration=0.0,
    thermochem=CrystalThermo(
        phonon_DOS=dos_fcc_emt,
        phonon_energies=omega_fcc_emt,
        formula_units=1,
        potentialenergy=E0_fcc,  # same DFT reference as Debye FCC
    ),
)

# Compare phonon DOS shapes
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
ax.plot(energies_debye * 1e3, dos_bcc_debye, label='α-Fe BCC')
ax.plot(energies_debye * 1e3, dos_fcc_debye, label='γ-Fe FCC')
ax.set_xlabel('Phonon energy (meV)')
ax.set_ylabel('DOS (1/eV)')
ax.set_title('Debye model')
ax.legend()

ax = axes[1]
ax.plot(omega_bcc_emt * 1e3, dos_bcc_emt, label='α-Fe BCC (EMT)')
ax.plot(omega_fcc_emt * 1e3, dos_fcc_emt, label='γ-Fe FCC (EMT)')
ax.set_xlabel('Phonon energy (meV)')
ax.set_ylabel('DOS (1/eV)')
ax.set_title('EMT force-constant calculation')
ax.legend()

plt.suptitle('Phonon density of states')
plt.tight_layout()

## 3  Helmholtz Free Energy

At $\Delta\mu = 0$ the semi-grand-canonical potential reduces to the Helmholtz free energy $F(T)$.
We compute the phase diagram for all four phases and use `plot_1d_T_phase_diagram` to visualise the
competing free energies.  Stable phases (lowest $F$ at each $T$) are shown with solid lines;
metastable phases with dashed lines.

In [ ]:
T = np.linspace(50, 1800, 200)

df = calc_phase_diagram(
    [bcc_debye, fcc_debye, bcc_emt, fcc_emt],
    Ts=T,
    mu=0.0,
    keep_unstable=True,
)

plot_1d_T_phase_diagram(df)
plt.title('Helmholtz free energies — Debye model and EMT phonons')

## 4  Transition Temperature

The equilibrium BCC → FCC transition occurs where the Debye free energies are equal.
`calc_phase_diagram` refines the transition to high precision and marks the corresponding rows with
`border == True`.  We run the Debye-only diagram to extract the unambiguous transition temperature.

In [ ]:
df_debye = calc_phase_diagram([bcc_debye, fcc_debye], Ts=T, mu=0.0)
border = df_debye.query('border')

if not border.empty:
    T_trans = border['T'].iloc[0]
    print(f'BCC → FCC transition temperature (Debye): {T_trans:.0f} K  ({T_trans - 273.15:.0f} °C)')
    print(f'Experimental value:                        1185 K  (912 °C)')

## Summary

| Quantity | Debye model | Experiment |
|----------|:-----------:|:----------:|
| $T_c$ (BCC → FCC) | ~1173 K | 1185 K |

The simple Debye model reproduces the BCC → FCC transition to within ~1% despite ignoring
magnetism and anharmonicity.

The EMT phonon calculation demonstrates the standard ASE workflow for computing a phonon DOS
from finite displacements.  Because EMT is not parameterised for Fe, the EMT free energies are
illustrative only; for production work the phonon DOS should come from a full force-constant
calculation with an appropriate potential (EAM, ML-FF) or with DFT via `phonopy`.